# M3 Agentic AI - Turning functions into tools

## 1. Introduction

### 1.1. Lab overview
In this lab you create a set of tools with **aisuite** and give them to an LLM. You'll see how the LLM requests tools, and how it chooses the right tools for its task.

### 🎯 1.2 Learning outcome
Apply tool-calling design patterns to agent workflows: give LLMs controlled access to Python functions via aisuite, manage parameter passing and execution flow, and validate multi-step outputs generated through tool orchestration.

> **Before you start:** copy `.env.example` (in the `ToolUseDemo` folder) to `.env` and add your `OPENAI_API_KEY`.

## 2. Setup: Initialize environment and client

In [ ]:
import json
import display_functions
from dotenv import load_dotenv
_ = load_dotenv()

### 2.1 Getting started with AISuite
Initialize the aisuite client: your interface for generating agent responses and calling tools.

In [ ]:
import aisuite as ai

# Create an instance of the AISuite client
client = ai.Client()

## 3. Build your first tool

### 3.1 Defining your function
A function that returns the current time as a string. The **docstring** matters: aisuite uses it to describe the tool to the LLM.

In [ ]:
from datetime import datetime

def get_current_time():
    """
    Returns the current time as a string.
    """
    return datetime.now().strftime("%H:%M:%S")

Test out your function to see exactly what it returns. (Locally this returns your local time.)

In [ ]:
get_current_time()

### 3.2 Turning your function into an LLM tool
First create the message structure: a list of messages, each with a `role` (`user`, `assistant`, `system`) and `content`.

In [ ]:
# Message structure
prompt = "What time is it?"
messages = [
    {
        "role": "user",
        "content": prompt,
    }
]

Then make the chat completion call:
- `model`: the model that will be used
- `messages`: the list of messages passed to the LLM
- `tools`: the list of tools the LLM has access to
- `max_turns`: the maximum number of turns the LLM may take, which prevents infinite tool-calling loops

In [ ]:
response = client.chat.completions.create(
    model="openai:gpt-4o",
    messages=messages,
    tools=[get_current_time],
    max_turns=5
)

# See the LLM response
print(response.choices[0].message.content)

### 3.3 Taking a closer look at the response
`pretty_print_chat_completion` extracts the steps from the response: the tool call the LLM requested, the result of running it on your machine, and the final answer.

In [ ]:
display_functions.pretty_print_chat_completion(response)

### 3.4 Manually defining tools
Behind the scenes, a tool is described to the LLM with a schema:
- `name`: the name of your local function
- `description`: what the function does, so the LLM knows when to use it
- `parameters`: the function's parameters (none here)

In [ ]:
tools = [{
    "type": "function",
    "function": {
        "name": "get_current_time", # <--- Your functions name
        "description": "Returns the current time as a string.", # <--- a description for the LLM
        "parameters": {}
    }
}]

When you define the schema yourself, aisuite expects **you** to handle the execution, so you don't use `max_turns`.

In [ ]:
response = client.chat.completions.create(
    model="openai:gpt-4o",
    messages=messages,
    tools=tools, # <-- Your list of tools with get_current_time
    # max_turns=5 # <-- When defining tools manually, you must handle calls yourself and cannot use max_turns
)

In [ ]:
print(json.dumps(response.model_dump(), indent=2, default=str))

Notice `tool_calls` under `message`: the LLM wants to call `get_current_time`. Run the tool locally, send the result back, and get the final response.

In [ ]:
response2 = None

# Create a condition in case tool_calls is in response object
if response.choices[0].message.tool_calls:
    # Pull out the specific tool metadata from the response
    tool_call = response.choices[0].message.tool_calls[0]
    args = json.loads(tool_call.function.arguments)

    # Run the tool locally
    tool_result = get_current_time()

    # Append the result to the messages list
    messages.append(response.choices[0].message)
    messages.append({
        "role": "tool", "tool_call_id": tool_call.id, "content": str(tool_result)
    })

    # Send the list of messages with the newly appended results back to the LLM
    response2 = client.chat.completions.create(
        model="openai:gpt-4o",
        messages=messages,
        tools=tools,
    )

    print(response2.choices[0].message.content)

## 4. Giving the LLM more tools

### 4.1 Three new tools
- **Weather tool** (`get_weather_from_ip`): finds your location from your IP address and returns the current, high and low temperature.
- **File writing tool** (`write_txt_file`): creates a text file with the given content (`file_path`, `content`).
- **QR code generator** (`generate_qr_code`): creates a QR code image with an embedded logo (`data`, `filename`, `image_path`).

In [ ]:
import requests
import qrcode
from qrcode.image.styledpil import StyledPilImage


def get_weather_from_ip():
    """
    Gets the current, high, and low temperature in Fahrenheit for the user's
    location and returns it to the user.
    """
    # Get location coordinates from the IP address
    lat, lon = requests.get('https://ipinfo.io/json').json()['loc'].split(',')

    # Set parameters for the weather API call
    params = {
        "latitude": lat,
        "longitude": lon,
        "current": "temperature_2m",
        "daily": "temperature_2m_max,temperature_2m_min",
        "temperature_unit": "fahrenheit",
        "timezone": "auto"
    }

    # Get weather data
    weather_data = requests.get("https://api.open-meteo.com/v1/forecast", params=params).json()

    # Format and return the simplified string
    return (
        f"Current: {weather_data['current']['temperature_2m']}°F, "
        f"High: {weather_data['daily']['temperature_2m_max'][0]}°F, "
        f"Low: {weather_data['daily']['temperature_2m_min'][0]}°F"
    )

# Write a text file
def write_txt_file(file_path: str, content: str):
    """
    Write a string into a .txt file (overwrites if exists).
    Args:
        file_path (str): Destination path.
        content (str): Text to write.
    Returns:
        str: Path to the written file.
    """
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(content)
    return file_path


# Create a QR code
def generate_qr_code(data: str, filename: str, image_path: str):
    """Generate a QR code image given data and an image path.

    Args:
        data: Text or URL to encode
        filename: Name for the output PNG file (without extension)
        image_path: Path to the image to be used in the QR code
    """
    qr = qrcode.QRCode(error_correction=qrcode.constants.ERROR_CORRECT_H)
    qr.add_data(data)

    img = qr.make_image(image_factory=StyledPilImage, embedded_image_path=image_path)
    output_file = f"{filename}.png"
    img.save(output_file)

    return f"QR code saved as {output_file} containing: {data[:50]}..."

### 4.2 Using your new tools
Now all the tools are passed to the LLM, and it chooses the appropriate one based on the prompt. Start with the weather tool.

In [ ]:
prompt = 'Can you get the weather for my location?'

response = client.chat.completions.create(
    model="openai:o4-mini",
    messages=[{"role": "user", "content": (
        prompt
    )}],
    tools=[
        get_current_time,
        get_weather_from_ip,
        write_txt_file,
        generate_qr_code
    ],
    max_turns=5
)

display_functions.pretty_print_chat_completion(response)

The LLM chose the correct tool even though it had access to the others. Now ask it to create a note.

In [ ]:
prompt = 'Can you make a txt note for me called reminders.txt that reminds me to call Daniel tomorrow at 7PM?'

response = client.chat.completions.create(
    model="openai:o4-mini",
    messages=[{"role": "user", "content": (
        prompt
    )}],
    tools=[
        get_current_time,
        get_weather_from_ip,
        write_txt_file,
        generate_qr_code
    ],
    max_turns=5
)

display_functions.pretty_print_chat_completion(response)

The LLM passed the right arguments, the tool ran locally, and the file now exists:

In [ ]:
with open('reminders.txt', 'r') as file:
    contents = file.read()
    print(contents)

Finally, make a QR code with an embedded logo (`dl_logo.jpg` is a placeholder logo in this folder; swap in your own).

In [ ]:
prompt = "Can you make a QR code for me using my company's logo that goes to www.deeplearning.ai? The logo is located at `dl_logo.jpg`. You can call it dl_qr_code."

response = client.chat.completions.create(
    model="openai:o4-mini",
    messages=[{"role": "user", "content": (
        prompt
    )}],
    tools=[
        get_current_time,
        get_weather_from_ip,
        write_txt_file,
        generate_qr_code
    ],
    max_turns=5
)

display_functions.pretty_print_chat_completion(response)

In [ ]:
from IPython.display import Image, display

# Display image directly
Image('dl_qr_code.png')

### 4.3 Using multiple tools at once
LLMs can chain several tool calls. In this prompt the note is requested *before* the weather is mentioned, but the LLM must call `get_weather_from_ip` first so it has content for `write_txt_file`.

🔍 **What to notice:**
- The LLM chooses which tool to use based on the request.
- Parameters (file names, content, URLs) are inferred from the message.
- Each tool's result is incorporated into the response.
- No-parameter tools (weather, time) are perfect for quick information requests.

In [ ]:
prompt = 'Can you help me create a qr code that goes to www.deeplearning.com from the image dl_logo.jpg? Also write me a txt note with the current weather please.'

response = client.chat.completions.create(
    model="openai:o4-mini",
    messages=[{"role": "user", "content": (
        prompt
    )}],
    tools=[
        get_weather_from_ip,
        get_current_time,
        write_txt_file,
        generate_qr_code
    ],
    max_turns=10
)

display_functions.pretty_print_chat_completion(response)

### Model Options
- `openai:gpt-4o`: optimized for reasoning and speed
- `openai:gpt-4.1`: strong reasoning, good for complex tasks
- `openai:gpt-4.1-mini`: lighter, faster and cheaper
- `openai:gpt-3.5-turbo`: efficient for simple tasks

Use smaller models for fast, low-cost prototyping; switch to stronger models for multi-step orchestration.

## Final Takeaways
- Tool calling lets LLMs go beyond text: they can use functions as part of their reasoning.
- Clear docstrings help the model know when and how to use each tool.
- aisuite turns Python functions into tool schemas and orchestrates multi-step workflows.
- Watching the flow (prompt → tool calls → results → final response) is essential for debugging agents.